# Marketing Campaign Impact on Purchases

You have a table of in-app purchases by user. Users that make their first in-app purchase are placed in a marketing campaign where they see call-to-actions for more in-app purchases. Find the number of users that made additional in-app purchases due to the success of the marketing campaign.

The marketing campaign doesn't start until one day after the initial in-app purchase so users that only made one or multiple purchases on the first day do not count, nor do we count users that over time purchase only the products they purchased on the first day.


🌀 Trust me, this one will surely challenge you...! You'll learn Cte, Joins, Group by. Give it a try and share the output! 👇

In [0]:
%skip
CREATE TABLE ska_catalog.bronze.in_app_purchases ( created_at TIMESTAMP, price BIGINT, product_id BIGINT, quantity BIGINT, user_id BIGINT);

INSERT INTO ska_catalog.bronze.in_app_purchases (created_at, price, product_id, quantity, user_id) VALUES('2024-12-01 10:00:00', 500, 101, 1, 1),  ('2024-12-02 11:00:00', 700, 102, 1, 1),('2024-12-01 12:00:00', 300, 103, 1, 2), ('2024-12-03 14:00:00', 400, 103, 1, 2),('2024-12-02 09:30:00', 200, 104, 1, 3), ('2024-12-04 15:30:00', 600, 105, 2, 3),('2024-12-01 08:00:00', 800, 106, 1, 4), ('2024-12-05 18:00:00', 500, 107, 1, 4),('2024-12-06 16:00:00', 700, 108, 1, 5); 

In [0]:
SELECT * FROM ska_catalog.bronze.in_app_purchases

In [0]:
WITH FristPurchase AS (
  SELECT
    user_id,
    MIN(created_at) AS first_purchase_date
  FROM
    ska_catalog.bronze.in_app_purchases
  GROUP BY
    user_id
)
SELECT
  COUNT(DISTINCT p.user_id) AS successful_users
FROM
  ska_catalog.bronze.in_app_purchases p
JOIN
  FristPurchase fp
ON p.user_id = fp.user_id
WHERE
  p.created_at > DATEADD(DAY, 1,fp.first_purchase_date)
  AND NOT EXISTS (
    SELECT 1
    FROM ska_catalog.bronze.in_app_purchases fp_products
    WHERE
      fp_products.user_id = p.user_id
      AND fp_products.product_id = p.product_id
      AND CAST(fp_products.created_at AS DATE) = CAST(fp.first_purchase_date AS DATE)
);